<a href="https://colab.research.google.com/github/jman4162/PyTorch-Vision-Transformers-ViT/blob/main/notebooks/Fine_tuning_Vision_Transformers_ViT_with_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tuning Vision Transformers (ViT) with PyTorch

Author: John Hodge

Date: 04/23/24

## Introduction

Vision Transformers (ViTs) have emerged as a powerful class of deep learning models for computer vision, rivaling traditional convolutional neural networks (CNNs) in various tasks. This tutorial demonstrates how to fine-tune the `vit_b_16` model for object classification using CIFAR-10.

### What is a Vision Transformer?

Vision Transformers are a class of deep learning models adapted from transformers, which were originally developed for natural language processing. ViTs apply the transformer's self-attention mechanism to grids of image patches, allowing the model to weigh the importance of different parts of an image. This ability to focus on relevant image features adaptively is particularly useful in complex visual recognition tasks.

The `vit_b_16` model, where "b" stands for "base" and "16" indicates the size of each image patch (16x16 pixels), is a medium-sized ViT model suitable for a wide range of vision tasks. It combines depth and complexity, offering a balanced trade-off between computational efficiency and accuracy.

![ViT](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/model_doc/vit_architecture.jpg)

Reference: [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)

### Tutorial Overview

This tutorial will guide you through the steps of fine-tuning the `vit_b_16` model using PyTorch. We will cover:

- **Setting up PyTorch and importing the ViT model**: How to load the pre-trained `vit_b_16` and prepare it for fine-tuning.
- **Data preparation**: Techniques for preparing your image data for training and evaluation, including data augmentation.
- **Fine-tuning process**: Adjustments and optimization for the model specific to object classification tasks.
- **Evaluation and testing**: How to assess the model's performance using accuracy, precision, recall, and confusion matrices.

By the end of this tutorial, you will have a solid understanding of how to implement and adapt Vision Transformers for real-world image classification tasks.

Let's dive into the world of Vision Transformers!

## Setup Environment
First, ensure you have Python installed, and then install PyTorch and torchvision. You can install them using pip:

In [ ]:
!pip install torch torchvision torchsummary tqdm gradio onnx onnxruntime

## Import Necessary Libraries

In [ ]:
import torch
import time
import random
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torch import nn
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, Subset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LambdaLR
from torch.amp import autocast, GradScaler
from torchvision.models import vit_b_16, vit_b_32, vit_l_16, ViT_B_16_Weights, ViT_B_32_Weights, ViT_L_16_Weights
from torchsummary import summary as model_summary
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print(f"Torch: {torch.__version__}")

Torch: 2.2.1+cu121


### Environment Setup (Colab vs Local)

The following cell detects whether you're running in Google Colab or locally, and sets up the appropriate model save directory.

In [ ]:
# Detect environment and set up model save directory
try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_DIR = '/content/drive/MyDrive/ViT_models/'
except ImportError:
    IN_COLAB = False
    MODEL_DIR = './models/'

os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Model save directory: {MODEL_DIR}")

### Set random seeds for repeatability

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

In [ ]:
seed = 42  # You can choose any integer value
seed_everything(seed)

## Data Preparation

We'll use the CIFAR-10 dataset, which contains 60,000 32x32 color images in 10 classes. Here's a detailed breakdown of each part:

### 1. Define Transformations

We use separate transforms for training and validation/test data. Training data gets augmentation (random flips, rotations, color jitter) to improve generalization, while validation/test data only gets resized and normalized.

```python
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
```
- **Components**:
  - `transforms.Resize((224, 224))`: Resizes each image to 224x224 pixels, which is the expected input size for Vision Transformers pretrained on ImageNet.
  - `transforms.RandomHorizontalFlip(p=0.5)`: Randomly flips images horizontally with 50% probability.
  - `transforms.RandomRotation(10)`: Randomly rotates images by up to 10 degrees.
  - `transforms.ColorJitter(...)`: Randomly adjusts brightness and contrast.
  - `transforms.ToTensor()`: Converts the images to PyTorch tensors.
  - `transforms.Normalize(...)`: Normalizes the image data to match ImageNet statistics.

### 2. Load Datasets with Proper Transforms

**Important**: We use `Subset` instead of `random_split` to ensure validation data uses the correct transforms (without augmentation). This is a common bug in many tutorials!

```python
# Load base dataset without transforms for splitting
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True)
indices = list(range(len(full_dataset)))

# Create train/val splits with PROPER transforms
train_dataset = Subset(
    datasets.CIFAR10(root='./data', train=True, transform=train_transform),
    train_indices
)
val_dataset = Subset(
    datasets.CIFAR10(root='./data', train=True, transform=val_transform),
    val_indices
)
```

### 3. Create DataLoaders
```python
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
```
- **Purpose**: DataLoaders efficiently manage batches of data during training and evaluation.

In [ ]:
# CIFAR-10 class names for later use
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                   'dog', 'frog', 'horse', 'ship', 'truck']

# Define separate transforms for training (with augmentation) and validation/test
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load base dataset to get indices for splitting
full_dataset = datasets.CIFAR10(root='./data', train=True, download=True)

# Create train/val split indices
train_size = int(0.8 * len(full_dataset))
indices = list(range(len(full_dataset)))
np.random.shuffle(indices)  # Shuffle for randomness (seeded)
train_indices = indices[:train_size]
val_indices = indices[train_size:]

# Create datasets with PROPER transforms using Subset
# This fixes the bug where random_split doesn't change transforms
train_dataset = Subset(
    datasets.CIFAR10(root='./data', train=True, download=False, transform=train_transform),
    train_indices
)
val_dataset = Subset(
    datasets.CIFAR10(root='./data', train=True, download=False, transform=val_transform),
    val_indices
)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

print(f"Train Data: {len(train_dataset)} (with augmentation)")
print(f"Validation Data: {len(val_dataset)} (no augmentation)")
print(f"Test Data: {len(test_dataset)} (no augmentation)")
print(f"Classes: {CIFAR10_CLASSES}")

## Model Setup: Define the Pretrained ViT Model for Fine-tuning

We'll use a pre-trained Vision Transformer and adapt it to our specific task (classifying 10 types of objects). 

### Available ViT Variants

We support multiple ViT variants, each with different trade-offs:

| Variant | Patch Size | Parameters | ImageNet Acc | Use Case |
|---------|------------|------------|--------------|----------|
| `vit_b_16` | 16x16 | 86M | 81.1% | **Recommended** - best accuracy/speed balance |
| `vit_b_32` | 32x32 | 88M | 75.9% | Faster inference, lower accuracy |
| `vit_l_16` | 16x16 | 304M | 79.7% | Larger model, needs more memory |

### Model Loading Function

```python
def load_vit_model(variant='vit_b_16', num_classes=10):
    model_fn, weights = VIT_VARIANTS[variant]
    model = model_fn(weights=weights)
    model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    return model
```

### Optional: torch.compile (PyTorch 2.x)

For PyTorch 2.0+, we can optionally compile the model for faster inference:
```python
if hasattr(torch, 'compile'):
    model = torch.compile(model, mode='reduce-overhead')
```

In [ ]:
# Available ViT variants with their weights
VIT_VARIANTS = {
    'vit_b_16': (vit_b_16, ViT_B_16_Weights.IMAGENET1K_V1),
    'vit_b_32': (vit_b_32, ViT_B_32_Weights.IMAGENET1K_V1),
    'vit_l_16': (vit_l_16, ViT_L_16_Weights.IMAGENET1K_V1),
}

def load_vit_model(variant='vit_b_16', num_classes=10, compile_model=False):
    """
    Load a pretrained ViT model and modify for the target number of classes.
    
    Args:
        variant: One of 'vit_b_16', 'vit_b_32', 'vit_l_16'
        num_classes: Number of output classes
        compile_model: Whether to use torch.compile (PyTorch 2.x)
    
    Returns:
        Configured ViT model
    """
    if variant not in VIT_VARIANTS:
        raise ValueError(f"Unknown variant: {variant}. Choose from {list(VIT_VARIANTS.keys())}")
    
    model_fn, weights = VIT_VARIANTS[variant]
    model = model_fn(weights=weights)
    
    # Modify the classification head for target number of classes
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    
    # Optional: Compile model for faster inference (PyTorch 2.x)
    if compile_model and hasattr(torch, 'compile'):
        print("Compiling model with torch.compile...")
        model = torch.compile(model, mode='reduce-overhead')
    
    return model

# Select variant and load model
VARIANT = 'vit_b_16'  # Change to 'vit_b_32' or 'vit_l_16' for comparison
USE_COMPILE = False  # Set to True for PyTorch 2.x optimization

model = load_vit_model(variant=VARIANT, num_classes=10, compile_model=USE_COMPILE)
print(f"Loaded {VARIANT} model with {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Define device and move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
model.to(device)
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### Define the loss function and the optimizer

We employ modern training techniques that are standard for transformer models:

#### 1. Loss Function
```python
criterion = nn.CrossEntropyLoss()
```
- **Purpose**: Evaluates the difference between the model's predictions and the actual labels.

#### 2. Optimizer: AdamW
```python
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=0.05)
```
- **Purpose**: AdamW (Adam with decoupled weight decay) is the standard optimizer for transformers, providing better regularization than vanilla Adam.

#### 3. Learning Rate Scheduler with Warmup
```python
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=5, eta_min=1e-6)
```
- **Purpose**: Cosine annealing provides smooth learning rate decay with periodic restarts, which helps escape local minima.

#### 4. Mixed Precision Training (AMP)
```python
scaler = GradScaler()
with autocast(device_type='cuda', dtype=torch.float16):
    outputs = model(images)
    loss = criterion(outputs, labels)
```
- **Purpose**: Automatic Mixed Precision (AMP) uses FP16 for most operations, providing 2-3x speedup with minimal accuracy loss.

#### 5. Learning Rate Warmup
```python
def get_warmup_lr(epoch, warmup_epochs=2, base_lr=1e-4):
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    return base_lr
```
- **Purpose**: Gradually increases learning rate during early epochs to stabilize training.

## Train the model

The `train_model` function is a comprehensive training loop with modern techniques:

### Key Features

1. **Mixed Precision Training (AMP)**: Uses FP16 for 2-3x speedup with `autocast` and `GradScaler`
2. **Gradient Clipping**: Prevents exploding gradients with `clip_grad_norm_`
3. **Learning Rate Warmup**: Gradually increases LR during early epochs
4. **Cosine Annealing**: Smooth LR decay with periodic restarts
5. **Early Stopping**: Stops training if validation loss doesn't improve for `patience` epochs
6. **Best Model Checkpointing**: Automatically saves the model with lowest validation loss
7. **Progress Tracking**: Real-time loss and LR display with tqdm

In [ ]:
# Training settings
batch_size = 64
epochs = 10
lr = 1e-4  # Higher base LR for AdamW
weight_decay = 0.05  # Standard weight decay for transformers
warmup_epochs = 2  # LR warmup epochs
patience = 3  # Early stopping patience
use_amp = True  # Enable mixed precision training

In [ ]:
def get_warmup_scheduler(optimizer, warmup_epochs, total_epochs):
    """Create a learning rate scheduler with linear warmup and cosine decay."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            # Linear warmup
            return (epoch + 1) / warmup_epochs
        else:
            # Cosine decay after warmup
            progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs=10, patience=3, use_amp=True):
    """
    Train the model with modern training techniques.
    
    Features:
    - Mixed precision training (AMP) for 2-3x speedup
    - Gradient clipping to prevent exploding gradients
    - Learning rate warmup with cosine decay
    - Early stopping and best model checkpointing
    
    Returns:
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        lr_history: List of learning rates per epoch
    """
    best_loss = float('inf')
    epochs_without_improvement = 0
    train_losses = []
    val_losses = []
    lr_history = []
    
    # Initialize gradient scaler for mixed precision
    scaler = GradScaler() if use_amp and device.type == 'cuda' else None
    amp_enabled = scaler is not None
    
    if amp_enabled:
        print("Mixed precision training enabled (AMP)")
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        start_time = time.time()
        current_lr = optimizer.param_groups[0]['lr']
        lr_history.append(current_lr)
        
        # Training phase with progress bar
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for images, labels in train_pbar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            
            # Mixed precision forward pass
            if amp_enabled:
                with autocast(device_type='cuda', dtype=torch.float16):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                # Scaled backward pass
                scaler.scale(loss).backward()
                
                # Unscale gradients and clip
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Optimizer step
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            
            running_loss += loss.item()
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.2e}'})
        
        epoch_time = time.time() - start_time
        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation phase
        model.eval()
        val_running_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                if amp_enabled:
                    with autocast(device_type='cuda', dtype=torch.float16):
                        outputs = model(images)
                        loss = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                val_running_loss += loss.item()

        avg_val_loss = val_running_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # Step scheduler
        scheduler.step()
        
        print(f'Epoch {epoch+1}: Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}, '
              f'LR: {current_lr:.2e}, Time: {epoch_time:.2f}s')

        # Check for improvement and save best model
        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            epochs_without_improvement = 0
            model_file_name = f'best_model_{VARIANT}_cifar10.pt'
            torch.save(model.state_dict(), model_file_name)
            torch.save(model.state_dict(), MODEL_DIR + model_file_name)
            print(f'New best model saved! (Val Loss: {best_loss:.6f})')
        else:
            epochs_without_improvement += 1
            print(f'No improvement for {epochs_without_improvement} epoch(s)')
            
        # Early stopping check
        if epochs_without_improvement >= patience:
            print(f'\nEarly stopping triggered after {epoch+1} epochs')
            break
    
    return train_losses, val_losses, lr_history

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Define loss function with AdamW optimizer and cosine scheduler with warmup
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = get_warmup_scheduler(optimizer, warmup_epochs=warmup_epochs, total_epochs=epochs)

print(f"Optimizer: AdamW (lr={lr}, weight_decay={weight_decay})")
print(f"Scheduler: Cosine with {warmup_epochs}-epoch warmup")

In [ ]:
# Train the model
train_losses, val_losses, lr_history = train_model(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    num_epochs=epochs, patience=patience, use_amp=use_amp
)

### Visualize Training Progress

Plot the training and validation loss curves to visualize how the model learned over time.

In [ ]:
# Plot training and validation loss with learning rate
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(range(1, len(train_losses) + 1), train_losses, 'b-', label='Training Loss', linewidth=2)
ax1.plot(range(1, len(val_losses) + 1), val_losses, 'r-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Learning rate plot
ax2.plot(range(1, len(lr_history) + 1), lr_history, 'g-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule (Warmup + Cosine Decay)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Evaluation

To evaluate the model, check its performance on the test dataset. The function `evaluate_model` computes the model's accuracy, which is the percentage of correctly predicted instances relative to the total number of instances evaluated. Here's a breakdown of how this function works:

### Function Definition
```python
def evaluate_model(model, test_loader):
```
- **Parameters**:
  - `model`: the neural network model that will be evaluated.
  - `test_loader`: a DataLoader object that provides batches of the test dataset, including both the input images and their corresponding labels.

### Set Model to Evaluation Mode
```python
model.eval()
```
- **Purpose**: This line sets the model to evaluation mode, which is crucial for models that have different behavior during training and testing, such as those using dropout layers or batch normalization. In evaluation mode, these layers will behave consistently and not apply randomness or scaling.

### Initialize Counters
```python
total = 0
correct = 0
```
- **Usage**:
  - `total`: keeps track of the total number of examples processed.
  - `correct`: counts the number of examples for which the model's prediction matches the actual label.

### Evaluation Loop
```python
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
```
- **Context**:
  - **`torch.no_grad()`**: This context manager disables gradient computation, reducing memory usage and speeding up the process since gradients are not needed for model evaluation.
  - **Data Movement**:
    - `images.to(device), labels.to(device)`: Moves the data to the appropriate computing device (CPU or GPU), which is necessary for models trained on GPUs.
  - **Model Prediction**:
    - `outputs = model(images)`: Feeds the batch of images into the model and gets the output logits for each class.
    - `_, predicted = torch.max(outputs.data, 1)`: Finds the predicted class label for each image by selecting the class with the highest logit value. The `torch.max` function returns both the maximum value and the index of that value (the predicted class label) across the specified dimension (`1`, meaning row-wise operation).
  - **Update Counters**:
    - `total += labels.size(0)`: Updates the total number of examples processed.
    - `correct += (predicted == labels).sum().item()`: Increases the count of correct predictions by the number of images in the current batch where the prediction matched the label.

### Calculate and Print Accuracy
```python
accuracy = 100 * correct / total
print(f'Accuracy on test images: {accuracy}%')
```
- **Calculation**:
  - Computes the percentage of correct predictions relative to the total number of predictions made.
- **Output**:
  - Prints the computed accuracy to provide feedback on how well the model is performing on the unseen test data.

This evaluation loop provides a straightforward and effective way to assess the accuracy of a model, allowing for the quantification of model performance in practical and operational terms.

### Run the evaluation loop on the best model

In [1]:
def evaluate_model(model, test_loader):
    model.eval()
    total = 0
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f'Accuracy on test images: {accuracy}%')

In [ ]:
# Load the best model
model_file_name = f'best_model_{VARIANT}_cifar10.pt'
best_model_state_dict = torch.load(model_file_name, map_location=device, weights_only=True)
model.load_state_dict(best_model_state_dict)
print(f"Loaded best model from {model_file_name}")

In [ ]:
# Evaluate the model
evaluate_model(model, test_loader)

Accuracy on test images: 97.65%


### Comprehensive Evaluation Metrics

Beyond simple accuracy, we'll compute precision, recall, and F1-score for each class, and visualize a confusion matrix.

In [ ]:
def get_predictions(model, test_loader):
    """Get all predictions and labels from the test set."""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Getting predictions'):
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_preds), np.array(all_labels)

# Get predictions
y_pred, y_true = get_predictions(model, test_loader)

In [ ]:
# Print classification report
print("Classification Report:")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=CIFAR10_CLASSES))

In [ ]:
# Create and plot confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - ViT on CIFAR-10')
plt.tight_layout()
plt.show()

### Results Analysis

The reported accuracy of "97.65%" on test images reflects a very good performance for the Vision Transformer model, particularly when considering the complexities and variabilities associated with image recognition tasks. Achieving such a high level of accuracy signifies that the model has effectively learned from the training data and can generalize well to new, unseen images, which is crucial for practical applications.

However, it is important to note that while this accuracy is commendable, it does not reach the state-of-the-art levels where accuracies exceed 99.5%. Such high-performance benchmarks are typically achieved by models that have undergone extensive fine-tuning on very large datasets and with substantial computational resources. These state-of-the-art models often involve:
- More complex architectures or ensemble methods that integrate outputs from multiple models to boost accuracy.
- Longer training times with numerous epochs, which allow the model to iteratively refine its weights and biases to better fit the data.
- Advanced regularization techniques and hyperparameter optimization strategies that can significantly improve model performance but require experimental tuning and computational power.

Reaching these top-tier accuracies usually demands considerable GPU compute power and time, making them less feasible within the constraints of a limited budget, as often is the case in educational or small-scale research settings. For the purposes of this tutorial, achieving an accuracy of 97.65% with the available resources and within a reasonable time frame is an impressive outcome. It demonstrates the capability of Vision Transformers to handle complex visual tasks effectively, offering a solid foundation for further exploration and optimization with more resources or in applications where very high accuracy is not the critical factor.

In summary, while the model does not achieve the pinnacle of current machine learning performance, it provides a robust and highly effective solution for many practical applications, especially where budget and computational resources are constrained.

State of the art (SOTA) benchmarks: [Image Classification on CIFAR-10
](https://paperswithcode.com/sota/image-classification-on-cifar-10)

## Attention Visualization

One of the key advantages of Vision Transformers is their interpretability through attention maps. We can visualize what parts of the image the model focuses on when making predictions.

### How Attention Works in ViT

1. The image is split into 14x14 = 196 patches (for 224x224 input with 16x16 patches)
2. Each patch attends to every other patch via self-attention
3. The CLS token aggregates information from all patches for classification
4. We can visualize which patches the CLS token attends to

The attention maps show "where the model is looking" to make its predictions.

In [ ]:
def get_attention_maps(model, image_tensor):
    """
    Extract attention maps from all layers of the ViT model.
    
    Args:
        model: ViT model
        image_tensor: Preprocessed image tensor [1, 3, 224, 224]
    
    Returns:
        attention_maps: List of attention tensors from each layer
    """
    attention_maps = []
    hooks = []
    
    def hook_fn(module, input, output):
        # output is a tuple: (attention_output, attention_weights)
        # For torchvision ViT, we need to capture attention differently
        attention_maps.append(output)
    
    # Register hooks on encoder layers
    for layer in model.encoder.layers:
        hook = layer.self_attention.register_forward_hook(hook_fn)
        hooks.append(hook)
    
    # Forward pass
    model.eval()
    with torch.no_grad():
        _ = model(image_tensor.to(device))
    
    # Remove hooks
    for hook in hooks:
        hook.remove()
    
    return attention_maps


def visualize_attention(model, image, layer_idx=-1, head_idx=None):
    """
    Visualize attention from the CLS token to image patches.
    
    Args:
        model: ViT model
        image: Original PIL image or tensor
        layer_idx: Which transformer layer to visualize (-1 for last)
        head_idx: Which attention head (None for average across heads)
    
    Returns:
        attention_map: 14x14 attention weights
    """
    # Prepare image
    if isinstance(image, Image.Image):
        input_tensor = val_transform(image).unsqueeze(0)
    else:
        input_tensor = image.unsqueeze(0) if image.dim() == 3 else image
    
    # Get attention maps by manual computation
    model.eval()
    
    # Get intermediate representations
    x = model._process_input(input_tensor.to(device))
    n = x.shape[0]
    
    # Expand the class token to the full batch
    batch_class_token = model.class_token.expand(n, -1, -1)
    x = torch.cat([batch_class_token, x], dim=1)
    
    # Pass through encoder layers
    for i, layer in enumerate(model.encoder.layers):
        if i == len(model.encoder.layers) + layer_idx if layer_idx < 0 else layer_idx:
            # Get attention weights from this layer
            # For manual computation, we'll use the layer's attention mechanism
            ln_x = layer.ln_1(x)
            
            # Compute attention manually to get weights
            qkv = layer.self_attention.in_proj_weight
            qkv_bias = layer.self_attention.in_proj_bias
            
            # Linear projection
            qkv_out = torch.nn.functional.linear(ln_x, qkv, qkv_bias)
            
            # Split into Q, K, V
            embed_dim = model.hidden_dim
            num_heads = layer.self_attention.num_heads
            head_dim = embed_dim // num_heads
            
            qkv_out = qkv_out.reshape(n, -1, 3, num_heads, head_dim).permute(2, 0, 3, 1, 4)
            q, k, v = qkv_out[0], qkv_out[1], qkv_out[2]
            
            # Compute attention weights
            scale = head_dim ** -0.5
            attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale
            attn_weights = torch.softmax(attn_weights, dim=-1)
            
            # Get CLS token attention (first token attending to all others)
            cls_attn = attn_weights[:, :, 0, 1:]  # [batch, heads, num_patches]
            
            if head_idx is not None:
                cls_attn = cls_attn[:, head_idx]  # [batch, num_patches]
            else:
                cls_attn = cls_attn.mean(dim=1)  # Average over heads
            
            # Reshape to 2D
            num_patches = int(cls_attn.shape[-1] ** 0.5)
            attn_map = cls_attn.reshape(num_patches, num_patches)
            
            return attn_map.cpu().numpy()
        
        # Continue forward pass
        x = layer(x)
    
    return None


def show_attention_on_image(image, attention_map, alpha=0.6):
    """
    Overlay attention map on the original image.
    
    Args:
        image: PIL Image
        attention_map: 14x14 numpy array
        alpha: Transparency for overlay
    """
    # Resize attention map to image size
    attn_resized = np.array(Image.fromarray(
        (attention_map * 255).astype(np.uint8)
    ).resize(image.size, Image.BILINEAR))
    
    # Normalize
    attn_resized = attn_resized / attn_resized.max()
    
    # Create heatmap
    heatmap = plt.cm.jet(attn_resized)[:, :, :3]
    heatmap = (heatmap * 255).astype(np.uint8)
    
    # Blend with original image
    img_array = np.array(image)
    blended = (1 - alpha) * img_array + alpha * heatmap
    blended = blended.astype(np.uint8)
    
    return blended


print("Attention visualization functions defined.")

In [ ]:
# Visualize attention on sample test images
def visualize_samples_with_attention(model, test_dataset, num_samples=4):
    """Show predictions with attention maps for random test samples."""
    
    # Get random samples
    indices = np.random.choice(len(test_dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))
    
    for i, idx in enumerate(indices):
        # Get image and label
        img_tensor, label = test_dataset[idx]
        
        # Convert tensor to PIL for visualization
        # Denormalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_denorm = img_tensor * std + mean
        img_denorm = torch.clamp(img_denorm, 0, 1)
        
        # Convert to PIL
        img_pil = transforms.ToPILImage()(img_denorm)
        
        # Get prediction
        model.eval()
        with torch.no_grad():
            output = model(img_tensor.unsqueeze(0).to(device))
            probs = torch.softmax(output, dim=1)
            pred_idx = output.argmax(dim=1).item()
            confidence = probs[0, pred_idx].item()
        
        # Get attention map
        attn_map = visualize_attention(model, img_tensor, layer_idx=-1)
        
        # Plot original image
        axes[i, 0].imshow(img_pil)
        axes[i, 0].set_title(f'True: {CIFAR10_CLASSES[label]}')
        axes[i, 0].axis('off')
        
        # Plot attention map
        if attn_map is not None:
            axes[i, 1].imshow(attn_map, cmap='hot')
            axes[i, 1].set_title('Attention Map (Last Layer)')
            axes[i, 1].axis('off')
            
            # Plot overlay
            overlay = show_attention_on_image(img_pil.resize((224, 224)), attn_map)
            axes[i, 2].imshow(overlay)
        else:
            axes[i, 1].text(0.5, 0.5, 'Attention extraction failed', ha='center', va='center')
            axes[i, 1].axis('off')
            axes[i, 2].imshow(img_pil)
        
        pred_label = CIFAR10_CLASSES[pred_idx]
        color = 'green' if pred_idx == label else 'red'
        axes[i, 2].set_title(f'Pred: {pred_label} ({confidence:.1%})', color=color)
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualize attention on test samples
visualize_samples_with_attention(model, test_dataset, num_samples=4)

### Visualize Misclassified Examples

Looking at failure cases helps understand model limitations:

## Model Deployment

This section covers exporting and optimizing the model for production deployment.

### ONNX Export

ONNX (Open Neural Network Exchange) allows running models in various runtimes:
- ONNX Runtime (CPU/GPU)
- TensorRT
- OpenVINO
- Web browsers (ONNX.js)

### Model Quantization

Quantization reduces model size and improves inference speed by using lower precision weights.

In [ ]:
import torch.onnx

def export_to_onnx(model, output_path="vit_cifar10.onnx"):
    """Export the model to ONNX format."""
    model.eval()
    model_cpu = model.cpu()
    
    # Create dummy input
    dummy_input = torch.randn(1, 3, 224, 224)
    
    # Export
    torch.onnx.export(
        model_cpu,
        dummy_input,
        output_path,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['image'],
        output_names=['logits'],
        dynamic_axes={
            'image': {0: 'batch_size'},
            'logits': {0: 'batch_size'}
        }
    )
    
    print(f"Model exported to {output_path}")
    
    # Verify the exported model
    import onnx
    onnx_model = onnx.load(output_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model validation passed!")
    
    # Get file size
    import os
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"Model size: {size_mb:.2f} MB")
    
    # Move model back to device
    model.to(device)
    
    return output_path

# Export to ONNX
onnx_path = export_to_onnx(model)

In [ ]:
# Test ONNX Runtime inference
import onnxruntime as ort

def test_onnx_inference(onnx_path, test_image):
    """Test inference with ONNX Runtime."""
    # Create session
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    
    # Get input name
    input_name = sess.get_inputs()[0].name
    
    # Prepare input
    if isinstance(test_image, torch.Tensor):
        input_data = test_image.numpy()
    else:
        input_data = test_image
    
    if input_data.ndim == 3:
        input_data = np.expand_dims(input_data, 0)
    
    # Run inference
    outputs = sess.run(None, {input_name: input_data.astype(np.float32)})
    logits = outputs[0]
    
    # Get prediction
    pred_idx = np.argmax(logits, axis=1)[0]
    probs = np.exp(logits) / np.sum(np.exp(logits), axis=1, keepdims=True)
    confidence = probs[0, pred_idx]
    
    return pred_idx, confidence

# Test ONNX inference on a sample
sample_img, sample_label = test_dataset[0]
pred_idx, conf = test_onnx_inference(onnx_path, sample_img)

print(f"ONNX Inference Test:")
print(f"  True label: {CIFAR10_CLASSES[sample_label]}")
print(f"  Predicted: {CIFAR10_CLASSES[pred_idx]} ({conf:.1%} confidence)")

### Inference Benchmarking

Compare inference speed across different deployment options:

## Interactive Demo with Gradio

Launch an interactive web interface to test the model with your own images.

**Note**: In Colab, this will create a shareable public link. Locally, it runs on localhost.

In [ ]:
import gradio as gr

def predict_image(image):
    """
    Predict class for an uploaded image.
    
    Args:
        image: PIL Image from Gradio
    
    Returns:
        Dictionary of class probabilities
    """
    if image is None:
        return {cls: 0.0 for cls in CIFAR10_CLASSES}
    
    # Preprocess
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]
    
    # Return as dictionary
    return {CIFAR10_CLASSES[i]: float(probs[i]) for i in range(10)}


# Create Gradio interface
demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload an Image"),
    outputs=gr.Label(num_top_classes=5, label="Predictions"),
    title="Vision Transformer Image Classifier",
    description="""
    Upload an image to classify it using a fine-tuned Vision Transformer (ViT).
    
    **Model**: vit_b_16 fine-tuned on CIFAR-10
    
    **Classes**: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck
    
    **Note**: This model was trained on 32x32 CIFAR-10 images. For best results, 
    use images of single objects similar to the training data.
    """,
    examples=[],  # Add example images if available
    theme="default"
)

# Launch the demo
# Set share=True in Colab to get a public URL
demo.launch(share=IN_COLAB, inline=True)

In [ ]:
def benchmark_inference(model, test_loader, num_batches=10):
    """Benchmark PyTorch inference speed."""
    model.eval()
    times = []
    
    with torch.no_grad():
        for i, (images, _) in enumerate(test_loader):
            if i >= num_batches:
                break
            
            images = images.to(device)
            
            # Warm up
            if i == 0:
                _ = model(images)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
            
            # Time inference
            start = time.time()
            _ = model(images)
            if device.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.time() - start)
    
    avg_time = np.mean(times) * 1000  # Convert to ms
    fps = batch_size / (avg_time / 1000)
    
    return avg_time, fps


def benchmark_onnx(onnx_path, test_loader, num_batches=10):
    """Benchmark ONNX Runtime inference speed."""
    sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    times = []
    
    for i, (images, _) in enumerate(test_loader):
        if i >= num_batches:
            break
        
        input_data = images.numpy().astype(np.float32)
        
        # Warm up
        if i == 0:
            _ = sess.run(None, {input_name: input_data})
        
        # Time inference
        start = time.time()
        _ = sess.run(None, {input_name: input_data})
        times.append(time.time() - start)
    
    avg_time = np.mean(times) * 1000
    fps = batch_size / (avg_time / 1000)
    
    return avg_time, fps


# Run benchmarks
print("Inference Benchmark Results")
print("=" * 50)

# PyTorch benchmark
pytorch_time, pytorch_fps = benchmark_inference(model, test_loader)
print(f"PyTorch ({device}):")
print(f"  Avg batch time: {pytorch_time:.2f} ms")
print(f"  Throughput: {pytorch_fps:.1f} images/sec")

# ONNX benchmark
onnx_time, onnx_fps = benchmark_onnx(onnx_path, test_loader)
print(f"\nONNX Runtime (CPU):")
print(f"  Avg batch time: {onnx_time:.2f} ms")
print(f"  Throughput: {onnx_fps:.1f} images/sec")

### Results Analysis

The reported accuracy of "97.65%" on test images reflects a very good performance for the Vision Transformer model, particularly when considering the complexities and variabilities associated with image recognition tasks. Achieving such a high level of accuracy signifies that the model has effectively learned from the training data and can generalize well to new, unseen images, which is crucial for practical applications.

However, it is important to note that while this accuracy is commendable, it does not reach the state-of-the-art levels where accuracies exceed 99.5%. Such high-performance benchmarks are typically achieved by models that have undergone extensive fine-tuning on very large datasets and with substantial computational resources. These state-of-the-art models often involve:
- More complex architectures or ensemble methods that integrate outputs from multiple models to boost accuracy.
- Longer training times with numerous epochs, which allow the model to iteratively refine its weights and biases to better fit the data.
- Advanced regularization techniques and hyperparameter optimization strategies that can significantly improve model performance but require experimental tuning and computational power.

Reaching these top-tier accuracies usually demands considerable GPU compute power and time, making them less feasible within the constraints of a limited budget, as often is the case in educational or small-scale research settings. For the purposes of this tutorial, achieving an accuracy of 97.65% with the available resources and within a reasonable time frame is an impressive outcome. It demonstrates the capability of Vision Transformers to handle complex visual tasks effectively, offering a solid foundation for further exploration and optimization with more resources or in applications where very high accuracy is not the critical factor.

In summary, while the model does not achieve the pinnacle of current machine learning performance, it provides a robust and highly effective solution for many practical applications, especially where budget and computational resources are constrained.

State of the art (SOTA) benchmarks: [Image Classification on CIFAR-10
](https://paperswithcode.com/sota/image-classification-on-cifar-10)

## Conclusions

This tutorial demonstrated modern techniques for fine-tuning Vision Transformers (ViT) for image classification using PyTorch, achieving **97.65% accuracy** on CIFAR-10.

### Key Features Covered

| Feature | Benefit |
|---------|---------|
| **Mixed Precision Training (AMP)** | 2-3x speedup on GPU |
| **AdamW + Cosine Annealing** | Better convergence for transformers |
| **Learning Rate Warmup** | Stable early training |
| **Proper Validation Transforms** | Fixed common `random_split` bug |
| **Multiple ViT Variants** | Choose accuracy vs. speed |
| **Attention Visualization** | Model interpretability |
| **ONNX Export** | Production deployment |
| **Gradio Demo** | Interactive testing |

### Key Takeaways

1. **Transfer Learning Works**: Pretrained ImageNet weights achieve 97.65% on CIFAR-10 in just a few epochs.

2. **Modern Training Matters**: AdamW, warmup, cosine scheduling, and AMP are standard for transformers.

3. **Data Handling is Critical**: Using `Subset` instead of `random_split` ensures validation data doesn't get augmentation.

4. **Interpretability**: Attention maps show what the model focuses on, building trust in predictions.

5. **Production Ready**: ONNX export enables deployment on various platforms.

### Next Steps

- Try larger variants (`vit_l_16`) for higher accuracy
- Experiment with CIFAR-100 (100 classes)
- Add custom datasets using `ImageFolder`
- Deploy with TensorRT or OpenVINO for faster inference
- Fine-tune with more epochs and larger learning rates

We hope this tutorial provides a solid, production-ready foundation for working with Vision Transformers!

## Additional Resources

- [Hugging Face ViT](https://huggingface.co/docs/transformers/en/model_doc/vit)
- [PyTorch ViT](https://pytorch.org/vision/main/models/vision_transformer.html)
- [D2L AI - Attention Mechanisms and Transformers](https://d2l.ai/chapter_attention-mechanisms-and-transformers/index.html)
- https://github.com/lucidrains/vit-pytorch